In [20]:
print("Everything is working fine!")

Everything is working fine!


### Chatbot

A chatbot is a software application designed to simulate human conversation through text or voice commands. They use Natural Language Processing (NLP) and artificial intelligence to understand user queries, offering instant, automated support on websites, messaging apps, and voice assistants


In [21]:
# Scanning the .env file for environment variables and loading them into the application
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
# import dependencies
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

In [8]:
# Ignore the warnings
import warnings
warnings.filterwarnings("ignore")

In [26]:
# Create an instance of the ChatGroq class, which will be used to interact with the Groq API for chatbot functionality
chat_groq = ChatGroq(model="qwen/qwen3-32b", temperature=0.7)

In [27]:
# Start a loop to continuously accept user input and generate responses from the chatbot until the user decides to exit or quit the application
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting the chatbot. Goodbye!")
        break
    response = chat_groq.invoke(user_input)
    result = StrOutputParser().invoke(response)
    print(f"ChatGroq: {result}")

ChatGroq: <think>
Okay, the user just said "Hi". I should respond friendly and ask how I can help them. Let me keep it simple and welcoming.
</think>

Hello! How can I assist you today?
ChatGroq: <think>
Okay, the user introduced themselves as Anant. I should greet them warmly. Let me make sure to use their name in the response to personalize it.

I need to ask how I can assist them today. Keep it friendly and open-ended. Maybe mention that I'm here for any questions or topics they want to explore. Let them know I can help with explanations, problem-solving, or just general knowledge. 

Wait, should I mention specific areas I'm good at? Like math, science, language help? Maybe include examples so they know the range of support. Also, remind them they can share their current projects or interests. 

Avoid using markdown, keep it natural. Use simple, conversational language. Make sure the tone is approachable and welcoming. 

Double-check for any errors. Alright, ready to respond.
</thin

**Observations**

1. Does not have the memory to hold the previus state
2. Identification of Role: User and Assistant

In [28]:
messages = [
    SystemMessage(content="You are a helpful assistant.")
]

while True:
    user_input = input("You: ")
    messages.append(HumanMessage(content=user_input))
    
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting the chatbot. Goodbye!")
        break
    
    results = chat_groq.invoke(messages)
    response = StrOutputParser().invoke(results)
    messages.append(AIMessage(content=response))
    print("Assistant:", response    )
print("Chat history:\n", messages)

Assistant: <think>
Okay, the user just said "Hi". That's a greeting. I should respond in a friendly and welcoming manner. Let me check if there's any specific context or previous conversation history that might affect my response. Since this is the start of a new conversation, I'll keep it general. Maybe ask how they're doing and offer assistance. Let me make sure the tone is positive and approachable.
</think>

Hello! How are you today? 😊 If you have any questions or need help with something, feel free to ask!
Assistant: <think>
Okay, the user wrote "Anant this sied" which might be a typo or a misunderstanding. They previously greeted "Hi" and I responded with a friendly hello. Now they mentioned "Anant this sied" which doesn't make much sense. Maybe they meant to say "Anant is this side" or "Anant, this side"? Or perhaps there's confusion with the name.

First, I should check if "Anant" is part of the message. Maybe they're trying to reach someone named Anant but I'm Qwen. The user m

**Observations**

1. I have used the simple list to store.
2. For long conversation it is problem.
3. Langchain has provided different kinds of memory to hold the chat history.

LangChain offers different memory classes depending on your needs:
- ConversationBufferMemory → Keeps the full history (like your list).
- ConversationBufferWindowMemory → Keeps only the last k turns.
- ConversationSummaryMemory → Summarizes past chats to keep context concise.
- ConversationKGMemory → Builds a knowledge graph of entities and relations.

Why use LangChain memory?
- Cleaner integration with chains and agents.
- Automatic handling of message types (HumanMessage, AIMessage).
- Easy switching between memory strategies (buffer, window, summary).
- Works seamlessly with ChatPromptTemplate.


In [24]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_groq import ChatGroq
from langchain_core.runnables import Runnable
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Create an instance of the ChatGroq class, which will be used to interact with the Groq API for chatbot functionality
model = ChatGroq(model="qwen/qwen3-32b", temperature=0.7)

In [25]:
# create instance of string output parser
parser = StrOutputParser()

In [ ]:
# Create chain using pipe operator
chain = model | StrOutputParser()

# Runnable can also be used to create a chain
# parser = StrOutputParser()
# chain = Runnable( model, parser)

In [30]:
# Prompt with memory
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use previous context when relevant."),
    ("user", "{input}")
])

# initialize the conversation with a system message
memory = ConversationBufferMemory(return_messages=True)
memory.chat_memory.add_ai_message("You are a helpful AI assistant. Use previous context when relevant.")

In [33]:
# Loop to continuously accept user input and generate responses from the chatbot until the user decides to exit or quit the application
while True:
    
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Exiting the chatbot. Goodbye!")
        break
    
    # Save the user input to memory
    memory.chat_memory.add_user_message(user_input)
    
    # Get full conversation history
    messages = memory.chat_memory.messages
    
     # Use pipe chain
    response = chain.invoke(messages)
    
    # Store AI response
    memory.chat_memory.add_ai_message(response)
    
    # print the response
    print(f"Assisstant: {response}")
    
print("Chat history:\n", memory.chat_memory.messages)

Assisstant: <think>
Okay, the user introduced themselves as Anant. I need to respond warmly and invite further interaction. Let me make sure to acknowledge their name and ask how I can assist them today. Keeping the tone friendly and open-ended seems best.
</think>

Hello Anant! Nice to meet you. How can I assist you today?
Assisstant: <think>
Okay, the user asked me to "Tell me about AI." Let me start by breaking down what they might be looking for. Since they introduced themselves as Anant, maybe they're a student or just someone curious about AI. I should explain AI in a clear and accessible way without too much jargon.

First, I'll define AI. It's the simulation of human intelligence by machines. Then I can mention the history, like when the term was coined in the 1950s. That gives context about its origins. Next, the different types: narrow AI versus general AI. People might not know the distinction, so that's important.

I should cover key areas where AI is applied—machine learni

Creating Streamlit application.